# Notebook 6 — Train, tune, evaluate

**Job of this notebook:** start with a simple baseline, train and tune a
real model against it, pick a metric that fits an imbalanced problem, and
touch the test set exactly once — at the very end.

**Reads:** `data/processed/features_{train,val,test}.parquet`.
**Writes:** `data/models/model.joblib`, `data/models/results_summary.json`.


In [ ]:
import sys
sys.path.append("../src")

import json
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    precision_recall_curve, classification_report, confusion_matrix,
)
import matplotlib.pyplot as plt

from config import (
    FEATURES_TRAIN_PATH, FEATURES_VAL_PATH, FEATURES_TEST_PATH,
    MODEL_PATH, RESULTS_PATH, LABEL_COL, RANDOM_STATE,
)

train = pd.read_parquet(FEATURES_TRAIN_PATH)
val = pd.read_parquet(FEATURES_VAL_PATH)
test = pd.read_parquet(FEATURES_TEST_PATH)

X_train, y_train = train.drop(columns=[LABEL_COL]), train[LABEL_COL]
X_val, y_val = val.drop(columns=[LABEL_COL]), val[LABEL_COL]
X_test, y_test = test.drop(columns=[LABEL_COL]), test[LABEL_COL]

print("late rate — train:", y_train.mean().round(3), " val:", y_val.mean().round(3), " test:", y_test.mean().round(3))


## 1. Metric choice

The late class is a small minority (single-digit percent, per Notebook 2).
**Accuracy is misleading here** — a model that always predicts "on time"
would score ~93%+ accuracy while catching zero late deliveries, which is
exactly the failure mode this model exists to avoid.

We use:
- **PR-AUC (average precision)** as the primary metric for tuning — it
  focuses on the positive (late) class and isn't inflated by the large
  negative class.
- **ROC-AUC** and **F1 on the late class** as secondary metrics to report
  alongside it.


## 2. Baseline — know what you have to beat

In [ ]:
baseline = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)

baseline_probs = baseline.predict_proba(X_val)[:, 1]
baseline_pr_auc = average_precision_score(y_val, baseline_probs)
baseline_roc_auc = roc_auc_score(y_val, baseline_probs)

print(f"Baseline (stratified random) — PR-AUC: {baseline_pr_auc:.4f}  ROC-AUC: {baseline_roc_auc:.4f}")

# A second, slightly stronger baseline: plain logistic regression, no tuning
logreg_baseline = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg_baseline.fit(X_train, y_train)
lr_probs = logreg_baseline.predict_proba(X_val)[:, 1]
print(f"Baseline (logistic regression) — PR-AUC: {average_precision_score(y_val, lr_probs):.4f}  "
      f"ROC-AUC: {roc_auc_score(y_val, lr_probs):.4f}")


## 3. Train and tune a stronger model using the validation split

Small manual grid search over a Random Forest, tuned against **validation**
PR-AUC. The test set stays untouched through all of this.


In [ ]:
param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [6, 12, None],
    "min_samples_leaf": [1, 5],
}

best_score = -1
best_params = None
best_model = None

for params in ParameterGrid(param_grid):
    model = RandomForestClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1,
        **params,
    )
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_val)[:, 1]
    score = average_precision_score(y_val, probs)
    print(f"{params} -> val PR-AUC={score:.4f}")
    if score > best_score:
        best_score, best_params, best_model = score, params, model

print()
print("Best params:", best_params, " best val PR-AUC:", round(best_score, 4))


## 4. Pick an operating threshold using validation

Default 0.5 rarely makes sense for an imbalanced problem. Use the
precision-recall curve on **validation** to pick a threshold, e.g. the one
that maximizes F1 on the late class — then apply that fixed threshold to
test in the next step (don't re-tune it on test).


In [ ]:
val_probs = best_model.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, val_probs)
f1s = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx = np.nanargmax(f1s[:-1])  # last point has no corresponding threshold
best_threshold = thresholds[best_idx]

print(f"Chosen threshold: {best_threshold:.3f}  "
      f"(val precision={precisions[best_idx]:.3f}, recall={recalls[best_idx]:.3f}, f1={f1s[best_idx]:.3f})")

plt.plot(recalls, precisions)
plt.scatter(recalls[best_idx], precisions[best_idx], color="red", zorder=5, label="chosen threshold")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curve (validation)")
plt.legend()
plt.show()


## 5. Touch the test set once, at the very end

This is the only cell in the whole project allowed to look at `y_test`.


In [ ]:
test_probs = best_model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)

test_pr_auc = average_precision_score(y_test, test_probs)
test_roc_auc = roc_auc_score(y_test, test_probs)
test_f1 = f1_score(y_test, test_preds)

print(f"TEST  PR-AUC: {test_pr_auc:.4f}   ROC-AUC: {test_roc_auc:.4f}   F1 (late class): {test_f1:.4f}")
print()
print(classification_report(y_test, test_preds, target_names=["on_time", "late"]))
print(confusion_matrix(y_test, test_preds))


## Artifacts: `model.joblib`, `results_summary.json`

In [ ]:
joblib.dump(best_model, MODEL_PATH)

results_summary = {
    "baseline_stratified_pr_auc_val": float(baseline_pr_auc),
    "baseline_logreg_pr_auc_val": float(average_precision_score(y_val, lr_probs)),
    "model": "RandomForestClassifier",
    "best_params": best_params,
    "chosen_threshold": float(best_threshold),
    "val_pr_auc": float(best_score),
    "test_pr_auc": float(test_pr_auc),
    "test_roc_auc": float(test_roc_auc),
    "test_f1_late_class": float(test_f1),
}

with open(RESULTS_PATH, "w") as f:
    json.dump(results_summary, f, indent=2)

print(f"Saved model to {MODEL_PATH}")
print(f"Saved results to {RESULTS_PATH}")
results_summary
